# E-Commerce Customer Purchase Prediction

**ShopSmart | Supervised binary classification | Logistic Regression**

This notebook follows the Sapphire IQ project brief: explore the supplied customer/session data, handle missing values and outliers, encode and scale predictors, train Logistic Regression, evaluate the requested metrics, and predict purchase likelihood for new example customers. `Purchase=1` means purchase; `Purchase=0` means no purchase. The customer ID is kept out of the model.

## 1. Load the data and inspect its quality

In [ ]:
from pathlib import Path
import sys
import pandas as pd
import matplotlib.pyplot as plt

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / 'src').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT))
DATA_PATH = PROJECT_ROOT / 'data' / 'Ecommerce_Purchase_Prediction_DataSet.csv'
data = pd.read_csv(DATA_PATH)
print(f'Shape: {data.shape[0]:,} rows × {data.shape[1]} columns')
display(data.head())
display(data.dtypes.to_frame('data_type'))
display(data.isna().sum().sort_values(ascending=False).to_frame('missing_values').query('missing_values > 0'))
print(f'Duplicate rows: {data.duplicated().sum()}')
display(data['Purchase'].value_counts().sort_index().rename(index={0: 'No purchase (0)', 1: 'Purchase (1)'}).to_frame('sessions'))

## 2. Explore purchase patterns

The target is moderately imbalanced (64.8% purchase sessions). Comparing purchase rates across groups is descriptive and should not be interpreted as a causal effect.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
data['Purchase'].value_counts().sort_index().plot(kind='bar', ax=axes[0], color=['#7A91A8', '#167D8D'])
axes[0].set(title='Purchase outcome', xlabel='Purchase label (0 / 1)', ylabel='Sessions')
missing = data.isna().sum().drop(labels=['Customer_ID', 'Purchase'], errors='ignore')
missing[missing > 0].sort_values().plot(kind='barh', ax=axes[1], color='#D99045')
axes[1].set(title='Missing values by field', xlabel='Missing cells')
data.groupby('Membership_Type')['Purchase'].mean().sort_values().plot(kind='barh', ax=axes[2], color='#167D8D')
axes[2].set(title='Purchase rate by membership', xlabel='Purchase rate (%)')
from matplotlib.ticker import PercentFormatter
axes[2].xaxis.set_major_formatter(PercentFormatter(1.0))
plt.tight_layout()
plt.show()
display(data.groupby('Membership_Type')['Purchase'].agg(['count', 'mean']).rename(columns={'mean': 'purchase_rate'}).sort_values('purchase_rate', ascending=False))

## 3. Preprocess, fit, and evaluate Logistic Regression

A stratified 80/20 split is used. Numeric missing values are filled with the median, numeric outliers are capped using IQR fences, and numeric features are standardized. Categorical missing values are filled with the most frequent category and categories are one-hot encoded. Each preprocessing step is fitted only on training data through a scikit-learn pipeline.

In [ ]:
from src.modeling import run_analysis
import json

metrics = run_analysis(DATA_PATH)
display(pd.DataFrame([metrics['metrics']]).T.rename(columns={0: 'test-set score'}).style.format('{:.3f}'))
print('Confusion matrix, actual rows and predicted columns (labels 0, 1):')
display(pd.DataFrame(metrics['confusion_matrix_labels_0_then_1'], index=['Actual 0', 'Actual 1'], columns=['Predicted 0', 'Predicted 1']))
print(f"Rows: {metrics['dataset_rows']:,}; training: {metrics['train_rows']:,}; test: {metrics['test_rows']:,}; threshold: {metrics['decision_threshold']}")
print('Evaluation chart:')
from IPython.display import Image, display
display(Image(filename=str(PROJECT_ROOT / 'reports' / 'figures' / 'model_evaluation.png')))

### Saved holdout results

| Accuracy | Precision | Recall | F1-score | ROC-AUC | Majority-class baseline |
|---:|---:|---:|---:|---:|---:|
| 70.35% | 72.60% | 87.11% | 79.20% | 0.740 | 64.80% |

Confusion matrix (actual rows, predicted columns; labels 0 then 1): `[[278, 426], [167, 1129]]`. The saved results come from the fixed stratified split with seed 42. Run the preceding cell to regenerate the metrics and chart.

![Holdout confusion matrix and ROC curve](../reports/figures/model_evaluation.png)

## 4. Example predictions for new customers

The probabilities below are illustrative model scores from three example profiles. They are not guarantees that a customer will purchase. The fitted pipeline and a CSV of these examples are saved under `models/` and `results/`.

In [ ]:
examples = pd.read_csv(PROJECT_ROOT / 'results' / 'sample_new_customer_predictions.csv')
display(examples[['Membership_Type', 'Device_Type', 'Session_Duration_Min', 'Cart_Items', 'Cart_Value_INR', 'purchase_probability', 'predicted_purchase']].style.format({'purchase_probability': '{:.1%}', 'Cart_Value_INR': '₹{:,.0f}'}))

## 5. Interpretation and limitations

The holdout recall is higher than precision at the 0.50 decision threshold, so the model identifies many actual purchasers but also flags some non-purchasers. Business teams should set a threshold based on campaign capacity and the relative cost of false positives and missed purchasers. Results are from one random holdout split; validate on later time periods before operational use. The dataset brief does not describe data provenance, so generalization to other stores or customer populations is unknown. Model coefficients and group rates are associations, not evidence of causation.

For the full written findings and figures, see `reports/Ecommerce_Customer_Purchase_Prediction_Report.pdf`.